In [14]:
# Load env variables and create client
from anthropic import Anthropic
from dotenv import load_dotenv
from IPython.display import display
from IPython.display import Markdown
import debugpy
import json
import os
import pprint, shutil
import textwrap

load_dotenv()

if not os.getenv("ANTHROPIC_API_KEY"):
    exit("No API key found in environment variables")

client = Anthropic()
model = "claude-haiku-4-5"

In [15]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.with_options(timeout=30.0).messages.create(**params)
    return message.content[0].text

def myprint(messages):
    width = shutil.get_terminal_size().columns
    for line in json.dumps(messages, indent=2, ensure_ascii=False).splitlines():
        indent = len(line) - len(line.lstrip())
        print(textwrap.fill(line, width=width, subsequent_indent=' ' * (indent + 2)))

In [16]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [17]:
dataset = generate_dataset()

myprint(dataset)

[
  {
    "task": "Write a Python function that takes an AWS S3 bucket name and
      returns True if it follows AWS naming conventions (lowercase, 3-63
      characters, no underscores), False otherwise."
  },
  {
    "task": "Create a JSON object that represents an AWS IAM policy allowing a
      principal to read and list objects from a specific S3 bucket named 'my-
      data-bucket'."
  },
  {
    "task": "Write a regular expression that matches valid AWS IAM user names
      (alphanumeric characters, hyphens, and underscores only, 1-64 characters
      long)."
  }
]


In [18]:
def run_prompt(test_case):
    '''Merges the prompt and test case input, then returns the result'''
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [19]:
def run_test_case(test_case) :
    '''Calls run_prompt, then grades the result'''
    output = run_prompt(test_case)

    # TODO - Grading
    score = 10

    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [20]:
def run_eval(dataset) :
    '''Loads the dataset and calls run_test_case with each case'''
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    return results

In [25]:
with open("dataset.json", "r") as f:
    dataset=json.load(f)

results=run_eval(dataset)

In [28]:
print(json.dumps(results, indent=2))
# myprint(results)

[
  {
    "output": "# AWS S3 Bucket Name Validator\n\nHere's a comprehensive solution with multiple approaches:\n\n## Solution 1: Single Regex (Recommended)\n\n```regex\n^(?!.*(-\\.|-{2}|\\.{2}|-$|\\.$))(?!.*\\.$)[a-z0-9]([a-z0-9\\.\\-]{1,61}[a-z0-9])?$\n```\n\n**Explanation:**\n- `^` - Start of string\n- `(?!.*(-\\.|-.{1,}\\.|-{2}|\\.{2}|-$|\\.$))` - Negative lookahead preventing:\n  - `(-\\.)` - Hyphen followed by period\n  - `-{2}` - Consecutive hyphens\n  - `\\.{2}` - Consecutive periods\n  - `-$` - Ending with hyphen\n  - `\\.$` - Ending with period\n- `[a-z0-9]` - Must start with letter or number\n- `([a-z0-9\\.\\-]{1,61}[a-z0-9])?` - Middle part (0-61 chars) + end with letter/number\n- `$` - End of string\n\n## Solution 2: JavaScript Implementation\n\n```javascript\nfunction isValidS3BucketName(bucketName) {\n  const regex = /^(?!.*(-\\.|-{2}|\\.{2}|-$|\\.$))(?!.*\\.$)[a-z0-9]([a-z0-9\\.\\-]{1,61}[a-z0-9])?$/;\n  return regex.test(bucketName) && bucketName.length >= 3 && bucket